---
## Section 0 — Configuration (Structured Streaming)

**Prérequis :** Session 2 §2.8 (Parquet consolidé) et ideally Session 3 (Delta).

Si vous ouvrez **Session 4 seule** (kernel vide), exécutez **cette cellule** avant les autres pour créer `spark`, les chemins (`STREAM_SOURCE_DIR`, `DELTA_*`, …) et les dossiers de streaming.


In [1]:
import os
import sys
import platform
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

# Mac Apple Silicon : Java arm64 + workers PySpark arm64 (identique Session 3)
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"
    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

# Chemins projet
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
OUTPUT_DIR = DATA_DIR / "output"
VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR = OUTPUT_DIR / "stream_input"
STREAM_CHECKPOINT = OUTPUT_DIR / "checkpoints"

assert VELIB_CONSOLIDE.exists(), (
    f"Fichier manquant : {VELIB_CONSOLIDE.resolve()}\n"
    "Exécutez Spark_DIA3_Session_2.ipynb §2.8 avant ce notebook."
)

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

APP_NAME = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8


def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def _delta_is_configured() -> bool:
    if not _spark_is_alive():
        return False
    try:
        extensions = spark.conf.get("spark.sql.extensions", "")
        catalog = spark.conf.get("spark.sql.catalog.spark_catalog", "")
        return (
            "DeltaSparkSessionExtension" in extensions
            and "DeltaCatalog" in catalog
        )
    except Exception:
        return False


def reset_spark_session() -> None:
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception as exc:
            print(f"spark.stop() ignoré : {exc}")

    if "sc" in globals() and sc is not None:
        try:
            if sc._jsc is not None:
                sc.stop()
        except Exception as exc:
            print(f"sc.stop() ignoré : {exc}")

    _clear_spark_session_registry()
    spark = None
    sc = None


def create_delta_spark_session():
    """Session Spark avec Delta Lake (obligatoire pour .format('delta'))."""
    global spark, sc

    reset_spark_session()

    _builder = (
        SparkSession.builder
        .appName(APP_NAME)
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()
    sc = spark.sparkContext
    sc.setLogLevel("WARN")

    if not _delta_is_configured():
        raise RuntimeError(
            "Delta Lake non actif. Kernel → Restart, puis relancez cette cellule."
        )

    print(f"Spark {spark.version} — Delta Lake activé")
    return spark, sc


if not _spark_is_alive() or not _delta_is_configured():
    create_delta_spark_session()
else:
    sc = spark.sparkContext
    print(f"Spark déjà actif ({spark.version}) — Delta Lake OK")

print(f"[OK] stream_input : {STREAM_SOURCE_DIR.resolve()}")


26/07/22 09:27:08 WARN Utils: Your hostname, MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.10.164.189 instead (on interface en0)
26/07/22 09:27:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/romain/.ivy2/cache
The jars for the packages stored in: /Users/romain/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fcb790d6-5074-4dab-9f4d-95fcc8655fb4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 60ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|   

:: loading settings :: url = jar:file:/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/07/22 09:27:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.9 — Delta Lake activé
[OK] stream_input : /Users/romain/Desktop/SparkVelib/data/output/stream_input


---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


In [2]:
# Vérification : le simulateur tourne-t-il ?
import time
from pathlib import Path

def compter_fichiers_stream(attente_sec: int = 5) -> list[Path]:
    """Attend et retourne les fichiers JSON produits par le simulateur."""
    time.sleep(attente_sec)
    return sorted(
        Path(STREAM_SOURCE_DIR).glob("*.json"),
        key=lambda p: p.stat().st_mtime,
    )

fichiers = compter_fichiers_stream(3)
n = len(fichiers)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print("  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    dernier = fichiers[-1]
    print(f"[OK] {n} fichier(s) JSON détecté(s).")
    print(f"  Dernier fichier : {dernier.name}")
    print("  Aperçu :")
    print(dernier.read_text(encoding="utf-8")[:400])


[OK] 161 fichier(s) JSON détecté(s).
  Dernier fichier : snapshot_000075.json
  Aperçu :
[{"station_id": 653099465, "nom_station": "Amédée Huon - Fort d'Ivry", "code_arr": 653099, "capacite": 31, "velos_meca": 1, "velos_elec": 1, "bornettes_libres": 29, "horodatage": "2020-11-26T18:47Z"}, {"station_id": 1810975577, "nom_station": "Guy Môquet - Parvis Georges Marchais", "code_arr": 1810975, "capacite": 30, "velos_meca": 0, "velos_elec": 1, "bornettes_libres": 29, "horodatage": "2020-11


---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [3]:
# Schéma du flux JSON produit par le simulateur
from pathlib import Path
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

if "STREAM_SOURCE_DIR" not in globals():
    _candidats = [Path("data"), Path("../data")]
    _data = next((p for p in _candidats if (p / "output").exists()), _candidats[0])
    STREAM_SOURCE_DIR = _data / "output" / "stream_input"

schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       StringType(),    False),
])

# Création du DataFrame de streaming — au plus 2 fichiers par micro-batch
stream_df = (
    spark.readStream
    .schema(schema_flux)
    .option("maxFilesPerTrigger", 2)
    .json(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame


Est un streaming DataFrame : True
Colonnes : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage']


In [4]:
# Ajout des colonnes calculées — exactement comme en batch
from pyspark.sql.functions import col, greatest, lit, round as spark_round, to_timestamp, when

stream_enrichi = (
    stream_df
    .withColumn("horodatage", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mm'Z'"))
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("bornettes_libres", greatest(col("bornettes_libres"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
    .withColumn(
        "taux_occupation",
        spark_round((col("capacite") - col("bornettes_libres")) / col("capacite"), 4),
    )
    .withColumn(
        "taux_occupation",
        when(col("taux_occupation") < 0, 0.0)
        .when(col("taux_occupation") > 1, 1.0)
        .otherwise(col("taux_occupation")),
    )
    .withColumn("est_vide", col("taux_occupation") < 0.10)
)

print("Colonnes après enrichissement :", stream_enrichi.columns)


Colonnes après enrichissement : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage', 'velos_total', 'taux_occupation', 'est_vide']


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [5]:
# Requête de streaming vers la console
import time
from pathlib import Path
from pyspark.sql.functions import col, greatest, lit, round as spark_round, when

# Enrichissement (même logique que la cellule précédente)
stream_console = (
    stream_df
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("bornettes_libres", greatest(col("bornettes_libres"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
    .withColumn(
        "taux_occupation",
        spark_round((col("capacite") - col("bornettes_libres")) / col("capacite"), 4),
    )
    .withColumn(
        "taux_occupation",
        when(col("taux_occupation") < 0, 0.0)
        .when(col("taux_occupation") > 1, 1.0)
        .otherwise(col("taux_occupation")),
    )
    .withColumn("est_vide", col("taux_occupation") < 0.10)
    .select(
        "station_id",
        "nom_station",
        "code_arr",
        "horodatage",
        "velos_total",
        "taux_occupation",
        "est_vide",
    )
)

_checkpoint_console = Path(STREAM_CHECKPOINT) / "console"
_checkpoint_console.mkdir(parents=True, exist_ok=True)

q_console = (
    stream_console
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", str(_checkpoint_console))
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(
    f"Lignes lues (dernier batch) : "
    f"{q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}"
)
q_console.stop()
print("Requête console arrêtée.")


26/07/22 09:32:20 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 20
-------------------------------------------
+----------+------------------------------------------+--------+-----------------+-----------+---------------+--------+
|station_id|nom_station                               |code_arr|horodatage       |velos_total|taux_occupation|est_vide|
+----------+------------------------------------------+--------+-----------------+-----------+---------------+--------+
|54000559  |Jouffroy d'Abbans - Wagram                |54000   |2020-11-26T19:05Z|2          |0.05           |true    |
|207633746 |Grande Armée - Brunel                     |207633  |2020-11-26T19:05Z|4          |0.0645         |true    |
|210566542 |Square Boucicaut                          |210566  |2020-11-26T19:05Z|3          |0.05           |true    |
|213682736 |Morillons - Dantzig                       |213682  |2020-11-26T19:05Z|5          |0.0962         |true    |
|653208733 |Romainville - Vaillant-Couturier          |653208 

In [45]:
# Équivalent Python pur (sans PySpark) — micro-batchs toutes les 5 s, max 2 fichiers
import json
import time
from pathlib import Path

stream_dir = Path(STREAM_SOURCE_DIR)

def enrichir_ligne(ligne: dict) -> dict | None:
    capacite = ligne.get("capacite")
    if not capacite or capacite <= 0:
        return None

    velos_meca = max(ligne.get("velos_meca") or 0, 0)
    velos_elec = max(ligne.get("velos_elec") or 0, 0)
    bornettes_libres = max(ligne.get("bornettes_libres") or 0, 0)
    velos_total = velos_meca + velos_elec

    taux = round((capacite - bornettes_libres) / capacite, 4)
    taux = max(0.0, min(1.0, taux))

    return {
        "station_id": ligne.get("station_id"),
        "nom_station": ligne.get("nom_station"),
        "code_arr": ligne.get("code_arr"),
        "horodatage": ligne.get("horodatage"),
        "velos_total": velos_total,
        "taux_occupation": taux,
        "est_vide": taux < 0.10,
    }

def lire_fichier_json(chemin: Path) -> list[dict]:
    donnees = json.loads(chemin.read_text(encoding="utf-8"))
    return donnees if isinstance(donnees, list) else [donnees]

def afficher_batch(numero: int, lignes: list[dict], max_lignes: int = 10) -> None:
    print("-" * 43)
    print(f"Batch: {numero}")
    print("-" * 43)
    if not lignes:
        print("(aucune ligne)")
        return
    cols = [
        "station_id", "nom_station", "code_arr", "horodatage",
        "velos_total", "taux_occupation", "est_vide",
    ]
    print(" | ".join(cols))
    for ligne in lignes[:max_lignes]:
        print(" | ".join(str(ligne[c]) for c in cols))
    if len(lignes) > max_lignes:
        print(f"only showing top {max_lignes} rows")

deja_vus: set[Path] = set()
batch_num = 0
fin = time.time() + 20

while time.time() < fin:
    time.sleep(5)

    fichiers = sorted(stream_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
    nouveaux = [f for f in fichiers if f not in deja_vus][:2]

    lignes_enrichies: list[dict] = []
    for chemin in nouveaux:
        deja_vus.add(chemin)
        for ligne in lire_fichier_json(chemin):
            enrichie = enrichir_ligne(ligne)
            if enrichie:
                lignes_enrichies.append(enrichie)

    afficher_batch(batch_num, lignes_enrichies)
    batch_num += 1

print(f"Micro-batchs affichés : {batch_num}")
print("Simulation console Python arrêtée.")


-------------------------------------------
Batch: 0
-------------------------------------------
station_id | nom_station | code_arr | horodatage | velos_total | taux_occupation | est_vide
54000559 | Jouffroy d'Abbans - Wagram | 54000 | 2020-11-26T19:05Z | 2 | 0.05 | True
207633746 | Grande Armée - Brunel | 207633 | 2020-11-26T19:05Z | 4 | 0.0645 | True
210566542 | Square Boucicaut | 210566 | 2020-11-26T19:05Z | 3 | 0.05 | True
213682736 | Morillons - Dantzig | 213682 | 2020-11-26T19:05Z | 5 | 0.0962 | True
653208733 | Romainville - Vaillant-Couturier | 653208 | 2020-11-26T19:05Z | 3 | 0.0789 | True
129094535 | Porte de Saint-Ouen - Bessières | 129094 | 2020-11-26T19:05Z | 4 | 0.0952 | True
54000612 | Cambrai - Benjamin Constant | 54000 | 2020-11-26T19:05Z | 4 | 0.093 | True
653132534 | Louis Lejeune - Barbès | 653132 | 2020-11-26T19:05Z | 2 | 0.08 | True
506455295 | Général Martial Valin - Pont du Garigliano | 506455 | 2020-11-26T19:05Z | 1 | 0.0625 | True
210408693 | Vaugirard - Mons

---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [46]:
from pyspark.sql.functions import avg, col, count, greatest, lit, to_timestamp, window

# Enrichissement intégré (évite un stream_enrichi obsolète en mémoire)
stream_pour_fenetres = (
    stream_df
    .withColumn("horodatage", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mm'Z'"))
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
)

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
df_fenetre_arr = (
    stream_pour_fenetres
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        col("code_arr"),
        window(col("horodatage"), "10 minutes", "2 minutes"),
    )
    .agg(
        avg("velos_total").alias("velos_moyens"),
        count("*").alias("nb_mesures"),
    )
    .select(
        col("code_arr"),
        col("window.start").alias("fenetre_debut"),
        col("window.end").alias("fenetre_fin"),
        col("velos_moyens"),
        col("nb_mesures"),
    )
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()


Schéma de la requête fenêtrée :
root
 |-- code_arr: integer (nullable = true)
 |-- fenetre_debut: timestamp (nullable = true)
 |-- fenetre_fin: timestamp (nullable = true)
 |-- velos_moyens: double (nullable = true)
 |-- nb_mesures: long (nullable = false)



In [ ]:
# Écriture vers Delta Lake (sink delta) -- mode append
import time
from pathlib import Path

if not _delta_is_configured():
    raise RuntimeError(
        "Delta Lake requis pour ce sink. Relancez la Section 0, puis readStream et fenêtres."
    )

path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
_checkpoint_fenetres = Path(STREAM_CHECKPOINT) / "fenetres_arr"
_checkpoint_fenetres.mkdir(parents=True, exist_ok=True)

q_fenetres = (
    df_fenetre_arr
    .writeStream
    .outputMode("append")
    .format("delta")
    .option("checkpointLocation", str(_checkpoint_fenetres))
    .trigger(processingTime="10 seconds")
    .queryName("fenetres_arrondissement")
    .start(path_fenetres)
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print(f"Sink Delta       : {path_fenetres}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")


Requête démarrée : fenetres_arrondissement
ID               : 961aaf45-73d4-412a-b18f-60fedb7b7aaf
Sink Delta       : data/output/delta/fenetres_arrondissement
En attente de 3 batchs...


26/07/21 16:43:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/21 16:43:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/21 16:43:50 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


KeyboardInterrupt: 

26/07/21 19:01:12 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 922113 milliseconds
26/07/21 21:43:38 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 923383 ms exceeds timeout 120000 ms
26/07/21 21:43:38 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/21 21:43:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzyc

In [ ]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    # TODO Lire les données au format delta, trier par début de fenêtre et en afficher 30
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")


---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [ ]:
# Table d'alertes : schema
schema_alertes = StructType([
    StructField("station_id",   IntegerType(),   False),
    StructField("nom_station",  StringType(),    True),
    StructField("code_arr",     IntegerType(),   True),
    StructField("debut_rupture",TimestampType(), True),
    StructField("fin_rupture",  TimestampType(), True),
    StructField("duree_min",    IntegerType(),   True),
    StructField("ts_alerte",    TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
# TODO Créer la table () partir d'un DataFrame)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")


In [ ]:
from delta.tables import DeltaTable

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[int, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None :
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    # TODO : Sortir di `batch_df` est vide

    # Agrégation au niveau station sur ce batch
    # On souhaite collecter :
    # - l'horodatage le plus ancien
    # - l'horodatage le plus récent
    # - le nombre d'observations où la station est vide
    # - le nombre total de stations
    # - le code de l'arrondissement
    # - le nom de la station
    etat_batch = (
        batch_df # ?
    ). # ??

    alertes = []
    # Examiner les transitions :
    # - non vide -> vide
    # - vide -> non vide
    # Dans le premier cas, ajouter la les informations dela station dans `etat_rupture`.
    # Dans le second cas :
    # - calculer le temps pendant le quel la station a été en rupture
    # - si ce temps est > 10 minutes, ajouter une alerte avec les informations détailllées
    # - retirer la station de la liste `etat_rupture`


    # Si des alertes ont été enregistrées,
    # les sauvegarder au format `delta` dans DELTA_ALERTES
    # Laisser une trace dans les logs


SyntaxError: invalid syntax (3973235338.py, line 30)

In [ ]:
# Lancement de la requête d'alerte
# - avec une tolérance de 5 minutes sur l'horodatage (cf. section suivante)
# - qui exécute la fonction `traiter_batch_alertes` pour chaque batch
# - définissant un `checkpoint` pour la sauvegarde de l'état en cas de panne
# - la requête s'appelle `q_alertes_rupture`
# - avec un intervalle de traitement de 10 ssecondes
q_alertes = (
    stream_enrichi # ?
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)


---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [ ]:
import json, datetime

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertroie surveillé par Spark.
# Chaque line est homogène à une observation (ou snapshot).
# N.B. Ajouter ne migne dans le fichier de logs
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    # TODO

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(15)

# Scénario 2 : retard de 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(15)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")


In [ ]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
q_exercice = None


---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [ ]:
# Arrêt de toutes les requêtes actives
actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

# TODO : Arrêter proprement les requêtes et reseigner le fichier logs

print("\nToutes les requêtes de streaming sont arrêtées.")


In [ ]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [ ]:
#spark.stop()
print("SparkSession arrêtée. À demain !")
